# RSNA Knee — Single-Click Swin-Base-384 Trainer
### One-Click Autonomous Pipeline: Run All & Walk Away
- Trains Swin-Base-384 on Fold 2
- Zero-manual configuration: Credentials and Gold-58 labels embedded
- Auto-saves and pushes weights to Kaggle Datasets via API


In [ ]:
# CELL 1: Environment Setup & Download Assets
!nvidia-smi
# Minimal clean install (pyarrow is pre-installed in Colab)
!pip install -U -q timm kaggle

import os, json, time, requests

# Authenticate Kaggle API for MRI corpus download
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': 'cyruspinto', 'key': 'KGAT_bf1b2382f2c6c8823e02df74217ba8e2'}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
print('[+] Kaggle API authenticated successfully.')

os.makedirs('/content/corpus', exist_ok=True)
os.makedirs('/content/labels', exist_ok=True)
os.makedirs('/content/rsna_comp', exist_ok=True)

# 1. Download Super Consensus Soft Labels via direct high-speed HTTPS (Bypasses Kaggle private token scopes)
print('Downloading soft labels (616 KB)...')
labels_url = 'https://files.catbox.moe/vk8q25.parquet'
r = requests.get(labels_url, headers={'User-Agent': 'Mozilla/5.0'})
with open('/content/labels/SUPER_CONSENSUS_labels.parquet', 'wb') as f:
    f.write(r.content)
print('[+] Soft labels downloaded successfully! Size:', os.path.getsize('/content/labels/SUPER_CONSENSUS_labels.parquet'), 'bytes')

# 2. Download 384px MRI corpus (public dataset dreaddevelopment/knee-raptor-corpus)
print('Downloading 384px MRI corpus from dreaddevelopment/knee-raptor-corpus...')
api.dataset_download_files('dreaddevelopment/knee-raptor-corpus', path='/content/corpus', unzip=True)
print('[+] Corpus downloaded:', os.listdir('/content/corpus'))

# 3. Write Gold-58 Ground Truth Anchor directly (0.001s)
gold_lines = [
    'StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker\'s,Contusion,Fracture',
    '1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0',
    '1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0',
    '1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0',
    '1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0',
    '1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0',
    '1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0',
    '1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0',
    '1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0',
    '1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0',
    '1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0',
    '1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0',
    '1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0',
    '1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0',
    '1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0',
    '1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0',
    '1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0',
    '1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0',
    '1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0',
    '1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0',
    '1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0',
    '1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0',
    '1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0',
    '1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0',
    '1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0',
    '1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0',
    '1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0',
    '1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0',
    '1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0',
    '1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0'
]
with open('/content/rsna_comp/train.csv', 'w') as f:
    f.write('\n'.join(gold_lines))

assert os.path.exists('/content/corpus/all_vols.npy'), 'Missing corpus all_vols.npy!'
assert os.path.exists('/content/labels/SUPER_CONSENSUS_labels.parquet'), 'Missing soft labels parquet!'
assert os.path.exists('/content/rsna_comp/train.csv'), 'Missing train.csv!'
print('[+] ALL DATA VALIDATED AND READY FOR TRAINING!')


In [ ]:
# CELL 2: Autonomous Swin-Base-384 Training Loop (Fold 2 & Fold 3)
import os, sys, math, time, gc, json
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

class TwoPartVols:
    def __init__(self, a, b=None):
        self.a, self.b = a, b
        self.len_a = len(a)
    def __getitem__(self, idx):
        if idx < self.len_a: return self.a[idx]
        return self.b[idx - self.len_a]
    def __len__(self):
        return self.len_a + (len(self.b) if self.b is not None else 0)

class SwinMILDataset(Dataset):
    def __init__(self, uids, id2row, labels, vols, res=384, train=True, n_slices=6):
        self.uids = uids
        self.id2row = id2row
        self.labels = labels
        self.vols = vols
        self.res = res
        self.train = train
        self.n_slices = n_slices
    def __len__(self): return len(self.uids)
    def __getitem__(self, idx):
        u = self.uids[idx]
        row = self.id2row[u]
        v = self.vols[row]
        D, H, W = v.shape
        step = max(1, D // self.n_slices)
        idxs = [min(i * step + step // 2, D - 1) for i in range(self.n_slices)]
        imgs = []
        for ci in idxs:
            c_prev = max(0, ci - 1)
            c_next = min(D - 1, ci + 1)
            triplet = np.stack([v[c_prev], v[ci], v[c_next]], axis=0).astype(np.float32) / 255.0
            imgs.append(triplet)
        t = torch.tensor(np.stack(imgs, axis=0), dtype=torch.float32)
        if (H, W) != (self.res, self.res):
            t = nn.functional.interpolate(t, size=(self.res, self.res), mode='bilinear', align_corners=False)
        y = torch.tensor(self.labels[u], dtype=torch.float32)
        return t, y

class SwinMILModel(nn.Module):
    def __init__(self, model_name='swin_base_patch4_window12_384.ms_in22k_ft_in1k', num_classes=12):
        super().__init__()
        self.encoder = timm.create_model(model_name, pretrained=True, num_classes=0)
        d = self.encoder.num_features
        self.attn = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, 1))
        self.classifier = nn.Linear(d, num_classes)
    def forward(self, x):
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)
        feats = self.encoder(x)
        feats = feats.view(B, S, -1)
        scores = self.attn(feats)
        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(feats * weights, dim=1)
        return self.classifier(pooled)

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Training on Device: {DEVICE}', flush=True)

# 1. Load 384px corpus
va = np.load('/content/corpus/all_vols.npy', mmap_mode='r')
vb = np.load('/content/corpus/extra_vols.npy', mmap_mode='r') if os.path.exists('/content/corpus/extra_vols.npy') else None
vols = TwoPartVols(va, vb)
a_ids = np.load('/content/corpus/all_ids.npy', allow_pickle=True).astype(str)
b_ids = np.load('/content/corpus/extra_ids.npy', allow_pickle=True).astype(str) if os.path.exists('/content/corpus/extra_ids.npy') else np.array([])
all_ids = np.concatenate([a_ids, b_ids]) if len(b_ids) > 0 else a_ids
id2row = {u: i for i, u in enumerate(all_ids)}
idset = set(all_ids)

# 2. Native Python Ground Truth Dictionary (Zero CSV/parser errors)
GOLD_LABELS = {'1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269': [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052': [0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238': [1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0], '1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840': [1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756': [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0], '1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604': [0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0], '1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581': [0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819': [1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0], '1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973': [1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492': [0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0], '1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], '1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891': [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], '1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903': [1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309': [0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192': [0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0], '1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0], '1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378': [1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0], '1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404': [0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645': [0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0], '1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924': [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608': [0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0], '1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911': [1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0], '1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945': [1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0], '1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742': [1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0], '1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0], '1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500': [0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125': [1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0], '1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187': [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0], '1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274': [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0], '1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450': [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363': [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0], '1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395': [1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0], '1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236': [1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0], '1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910': [0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039': [0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0], '1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327': [0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774': [1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921': [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0], '1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995': [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190': [1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0], '1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003': [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0], '1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828': [0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254': [1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037': [1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435': [1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], '1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662': [1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0]}
gold_ids = [u for u in GOLD_LABELS.keys() if u in idset]
print(f'[+] Gold-58 labels matched: {len(gold_ids)}/{len(GOLD_LABELS)}', flush=True)

# 3. Load Super Consensus Soft Labels
soft = pd.read_parquet('/content/labels/SUPER_CONSENSUS_labels.parquet').set_index('StudyInstanceUID')
train_ids = [u for u in all_ids if u in soft.index and u not in set(gold_ids)]
labels = {u: soft.loc[u, TARGETS].values.astype(np.float32) for u in train_ids}
for u in gold_ids:
    labels[u] = np.array(GOLD_LABELS[u], dtype=np.float32)

print(f'[+] Total training pool: {len(train_ids)} soft + {len(gold_ids)} gold = {len(labels)} total', flush=True)

np.random.seed(42)
shuffled_train = list(train_ids)
np.random.shuffle(shuffled_train)
n_val = len(shuffled_train) // 5

FOLDS_TO_TRAIN = [2, 3]
EPOCHS = 6
BATCH_SIZE = 8

for fold in FOLDS_TO_TRAIN:
    print(f'\n' + '='*60)
    print(f'=== STARTING ENGINE 10 SWIN-BASE-384: FOLD {fold} ===')
    print('='*60, flush=True)
    val_ids = shuffled_train[fold * n_val : (fold + 1) * n_val]
    core_train_ids = [u for u in shuffled_train if u not in set(val_ids)]
    
    tds = SwinMILDataset(core_train_ids, id2row, labels, vols, res=384, train=True, n_slices=6)
    tl = DataLoader(tds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
    
    model = SwinMILModel().to(DEVICE)
    optimizer = torch.optim.AdamW([
        {'params': model.encoder.parameters(), 'lr': 3e-5, 'weight_decay': 1e-4},
        {'params': model.attn.parameters(), 'lr': 3e-4, 'weight_decay': 1e-4},
        {'params': model.classifier.parameters(), 'lr': 3e-4, 'weight_decay': 1e-4}
    ])
    criterion = nn.BCEWithLogitsLoss()
    scaler = torch.cuda.amp.GradScaler()
    
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        start_t = time.time()
        for step, (bx, by) in enumerate(tl):
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                logits = model(bx)
                loss = criterion(logits, by)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            if (step + 1) % 50 == 0:
                print(f'Fold {fold} | Epoch {epoch+1}/{EPOCHS} | Step {step+1}/{len(tl)} | Loss: {total_loss/(step+1):.4f}', flush=True)
        elapsed_epoch = time.time() - start_t
        print(f'Epoch {epoch+1} finished in {elapsed_epoch:.1f}s. Avg Loss: {total_loss/len(tl):.4f}', flush=True)
        
    weight_path = f'/content/engine10_swin_384_fold{fold}_best.pt'
    torch.save(model.state_dict(), weight_path)
    print(f'[+] Saved model weights to {weight_path}', flush=True)
    
    # Export directly to Kaggle Datasets via API
    print(f'Pushing Fold {fold} weights to Kaggle Datasets...', flush=True)
    pkg_dir = f'/content/kaggle_engine10_fold{fold}'
    os.makedirs(pkg_dir, exist_ok=True)
    os.system(f'cp {weight_path} {pkg_dir}/')
    meta = {
        'title': f'RSNA Knee Engine 10 Swin-Base-384 Fold {fold} Weights',
        'id': f'cyruspinto/rsna-knee-engine10-fold{fold}-weights',
        'licenses': [{'name': 'CC0-1.0'}]
    }
    with open(f'{pkg_dir}/dataset-metadata.json', 'w') as f:
        json.dump(meta, f, indent=2)
    try:
        api.dataset_create_new(pkg_dir, dir_mode='zip')
        print(f'[+] Created new dataset: cyruspinto/rsna-knee-engine10-fold{fold}-weights')
    except Exception as e:
        print(f'Dataset exists, creating version update...')
        api.dataset_create_version(pkg_dir, f'Update Fold {fold} Weights', dir_mode='zip')
        print(f'[+] Updated dataset: cyruspinto/rsna-knee-engine10-fold{fold}-weights')
    
    del model, optimizer
    torch.cuda.empty_cache()
    gc.collect()

print('\n' + '='*60)
print('=== TRAINING AND UPLOAD COMPLETE! ===')
print('='*60, flush=True)

